# Stage 0 — Foundation

Shared substrate for the recommender pipeline (`STAGE0_FOUNDATION.md`).

**Run all cells top-to-bottom** before `content_based_model.ipynb` (same kernel).


## Cell 0-1 — Imports, paths, config


In [15]:
import hashlib
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import ndcg_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# Notebook-safe project root
ROOT = Path.cwd()
if not (ROOT / "Speed Dating Data.csv").exists():
    for _c in [ROOT, ROOT.parent]:
        if (_c / "Speed Dating Data.csv").exists():
            ROOT = _c
            break

RAW_CSV = ROOT / "Speed Dating Data.csv"
CLEAN_PARQUET = ROOT / "speed_dating_clean.parquet"
if not CLEAN_PARQUET.exists():
    alt = ROOT / "kaggle_upload" / "speed_dating_clean.parquet"
    if alt.exists():
        CLEAN_PARQUET = alt
TEAM_DIR = ROOT / "Teammates' code" / "CS608_Speed_dating_Recomm_System-main"
RESULTS = ROOT / "results"
for sub in ["runs", "explanations", "stability", "figures"]:
    (RESULTS / sub).mkdir(parents=True, exist_ok=True)

SEED = 42

CONFIG = {
    "s1_C": 1.0,
    "s2_k": 100,
    "fm_k": 8,
    "fm_lr": 0.05,
    "fm_epochs": 400,
    "fm_l2": 1e-4,
    "fm_patience": 40,
    "seed": SEED,
    "metrics_version": "v1-experiments_splits-compatible",
    "FROZEN": False,
}
print("Config loaded. FROZEN =", CONFIG["FROZEN"])


Config loaded. FROZEN = False


## Cell 0-2 — Provenance utilities


In [16]:
def file_sha(path: Path, n_hex: int = 12) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()[:n_hex]


def df_fingerprint(df: pd.DataFrame) -> dict:
    return {
        "n_rows": int(len(df)),
        "n_waves": int(df["wave"].nunique()),
        "waves": sorted(int(w) for w in df["wave"].unique()),
        "dec_rate": round(float(df["dec"].mean()), 4),
        "match_rate": round(float(df["match"].mean()), 4),
    }


def params_hash(obj) -> str:
    return hashlib.sha256(
        json.dumps(obj, sort_keys=True, default=str).encode()
    ).hexdigest()[:12]


def write_run_record(scheme: str, split_tag: str, record: dict) -> Path:
    ts = time.strftime("%Y%m%d_%H%M%S")
    path = RESULTS / "runs" / f"{scheme}_{split_tag}_{ts}.json"
    path.write_text(json.dumps(record, indent=2, default=str), encoding="utf-8")
    log_path = RESULTS / "experiment_log.csv"
    line = {
        "ts": ts,
        "scheme": scheme,
        "split": split_tag,
        "data_sha": record.get("data_sha", ""),
        "n_train": record.get("n_train", ""),
        "n_eval": record.get("n_eval", ""),
        **{
            k: round(v, 4)
            for k, v in record.get("metrics", {}).items()
            if isinstance(v, (int, float))
        },
    }
    pd.DataFrame([line]).to_csv(
        log_path, mode="a", header=not log_path.exists(), index=False
    )
    return path


print("Provenance utils ready.")


Provenance utils ready.


## Cell 0-3 — Load data, wave-12 reconstruction, two-sided table


In [17]:
INTEREST_COLS = [
    "sports", "tvsports", "exercise", "dining", "museums", "art",
    "hiking", "gaming", "clubbing", "reading", "tv", "theater",
    "movies", "concerts", "music", "shopping", "yoga",
]
OPP_BELIEF_COLS = ["attr2_1", "sinc2_1", "intel2_1", "fun2_1", "amb2_1", "shar2_1"]
SELF_COLS = ["attr3_1", "sinc3_1", "intel3_1", "fun3_1", "amb3_1"]
NORM_COLS = [
    "attr1_1_norm", "sinc1_1_norm", "intel1_1_norm",
    "fun1_1_norm", "amb1_1_norm", "shar1_1_norm",
]
PREF_RAW = ["attr1_1", "sinc1_1", "intel1_1", "fun1_1", "amb1_1", "shar1_1"]
ID_COLS = [
    "iid", "pid", "gender", "wave", "condtn", "round",
    "position", "order", "dec", "dec_o", "match",
]


def _recode_met(s):
    out = pd.Series(np.nan, index=s.index)
    out[s == 1] = 1
    out[s.isin([0, 2])] = 0
    return out


def _derive_clean(raw_rows, clean_cols):
    df = raw_rows.copy()
    df = df[df["pid"].notna()].copy()
    df[INTEREST_COLS] = df[INTEREST_COLS].clip(upper=10)
    df["met_before"] = _recode_met(df["met"])
    df["met_before_o"] = _recode_met(df["met_o"])
    df["scale_type"] = np.where(df["wave"].isin([6, 7, 8, 9]), "1-10", "100pt")
    psum = df[PREF_RAW].sum(axis=1, min_count=6)
    valid = psum > 0
    pref_norm = [c.replace("1_1", "1_1_norm") for c in PREF_RAW]
    df[pref_norm] = np.nan
    df.loc[valid, pref_norm] = df.loc[valid, PREF_RAW].div(psum[valid], axis=0).values
    df["age_diff"] = (df["age"] - df["age_o"]).abs()
    both = df["race"].notna() & df["race_o"].notna()
    df["race_match"] = np.where(both, (df["race"] == df["race_o"]).astype(float), np.nan)
    return df.reindex(columns=clean_cols)


def _validate_reconstruction(clean_df, raw, check_wave=11):
    cat_cols = ["race", "race_o", "field_cd", "career_c", "goal", "scale_type"]
    cont = (
        ["age", "age_o", "imprace", "imprelig", "exphappy", "int_corr", "date", "go_out", "age_diff"]
        + INTEREST_COLS + OPP_BELIEF_COLS
    )
    bin_cols = ["samerace", "met_before", "met_before_o", "race_match"]
    check_cols = ID_COLS + cont + SELF_COLS + NORM_COLS + cat_cols + bin_cols
    recon = _derive_clean(raw[raw["wave"] == check_wave], clean_df.columns)
    actual = clean_df[clean_df["wave"] == check_wave]
    recon = recon.sort_values(["iid", "pid"]).reset_index(drop=True)
    actual = actual.sort_values(["iid", "pid"]).reset_index(drop=True)
    assert len(recon) == len(actual), "row count mismatch on validation wave"
    bad = []
    for c in check_cols:
        a, b = actual[c], recon[c]
        if pd.api.types.is_numeric_dtype(a):
            if (a.fillna(-9e9) - b.fillna(-9e9)).abs().max() > 1e-9:
                bad.append(c)
        elif (a.fillna("NA").astype(str) != b.fillna("NA").astype(str)).any():
            bad.append(c)
    assert not bad, f"Reconstruction validation FAILED on: {bad}"


def _row_cosine(a: np.ndarray, b: np.ndarray, min_dims: int = 8) -> np.ndarray:
    mask = ~(np.isnan(a) | np.isnan(b))
    a0, b0 = np.where(mask, a, 0.0), np.where(mask, b, 0.0)
    num = (a0 * b0).sum(1)
    den = np.sqrt((a0 ** 2).sum(1)) * np.sqrt((b0 ** 2).sum(1))
    out = np.where(den > 0, num / np.maximum(den, 1e-12), np.nan)
    return np.where(mask.sum(1) >= min_dims, out, np.nan)


PROFILE_JOIN = SELF_COLS + INTEREST_COLS + NORM_COLS + ["gender"]


def build_two_sided(clean_df: pd.DataFrame) -> pd.DataFrame:
    prof = clean_df.drop_duplicates("iid")[["iid"] + PROFILE_JOIN].copy()
    out = clean_df.merge(
        prof.rename(columns={c: f"{c}_B" for c in PROFILE_JOIN}),
        left_on="pid",
        right_on="iid",
        how="left",
        suffixes=("", "_dropme"),
    )
    out = out.drop(columns=[c for c in out.columns if c.endswith("_dropme")])
    out["int_cos"] = _row_cosine(
        out[INTEREST_COLS].to_numpy(float),
        out[[f"{c}_B" for c in INTEREST_COLS]].to_numpy(float),
    )
    return out


clean20 = pd.read_parquet(CLEAN_PARQUET)
raw = pd.read_csv(RAW_CSV, encoding="latin-1").rename(columns={"Unnamed: 0": "iid"})
_validate_reconstruction(clean20, raw, check_wave=11)
print("Wave reconstruction gate: PASSED")

w12 = _derive_clean(raw[raw["wave"] == 12], clean20.columns)
clean21 = pd.concat([clean20, w12], ignore_index=True)

DATA = {
    "drop": build_two_sided(clean20),
    "keep": build_two_sided(clean21),
}
DATA_SHA = {"clean_parquet": file_sha(CLEAN_PARQUET), "raw_csv": file_sha(RAW_CSV)}

for k, d in DATA.items():
    fp = df_fingerprint(d)
    print(
        f"{k}: rows={fp['n_rows']:,}  waves={fp['n_waves']}  "
        f"dec={fp['dec_rate']}  match={fp['match_rate']}"
    )



Wave reconstruction gate: PASSED
drop: rows=7,976  waves=20  dec=0.424  match=0.1678
keep: rows=8,368  waves=21  dec=0.4201  match=0.1649


## Cell 0-4 — Feature lists & theme map


In [18]:
CONT_COLS = (
    ["age", "age_o", "imprace", "imprelig", "exphappy", "int_corr", "date", "go_out", "age_diff"]
    + INTEREST_COLS + OPP_BELIEF_COLS
)
BIN_COLS = ["samerace", "met_before", "met_before_o", "race_match"]
SELF_Z = [c + "_z" for c in SELF_COLS]
SELF_B = [f"{c}_B" for c in SELF_COLS]
SELF_B_Z = [c + "_z" for c in SELF_B]
INTEREST_B = [f"{c}_B" for c in INTEREST_COLS]
NORM_B = [f"{c}_B" for c in NORM_COLS]

TRAITS = ["attr", "sinc", "intel", "fun", "amb"]
ALIGN_COLS = [f"align_{t}" for t in TRAITS] + ["align_shar"]

S1_FEATS = (
    ALIGN_COLS + NORM_COLS + SELF_B_Z
    + [
        "int_cos", "int_corr", "age_diff", "samerace", "race_match",
        "imprace", "imprelig", "date", "go_out", "exphappy",
        "met_before", "met_before_o",
    ]
)

KNN_COLS = (
    CONT_COLS + SELF_Z + NORM_COLS + BIN_COLS
    + SELF_B_Z + INTEREST_B + NORM_B + ["int_cos"]
)

CTX_COLS = [
    "age_diff", "samerace", "race_match", "met_before", "met_before_o",
    "int_corr", "int_cos", "exphappy", "scale_type_code", "condtn",
]

THEME_MAP = {
    **{f"align_{t}": th for t, th in zip(
        TRAITS, ["Attractiveness", "Sincerity", "Intelligence", "Fun", "Ambition"])},
    "align_shar": "Shared interests",
    **{c: th for c, th in zip(
        NORM_COLS, ["Attractiveness", "Sincerity", "Intelligence", "Fun", "Ambition", "Shared interests"])},
    **{c: th for c, th in zip(
        SELF_B_Z, ["Attractiveness", "Sincerity", "Intelligence", "Fun", "Ambition"])},
    "int_cos": "Shared interests",
    "int_corr": "Shared interests",
    "age_diff": "Demographics fit",
    "samerace": "Demographics fit",
    "race_match": "Demographics fit",
    "imprace": "Demographics fit",
    "imprelig": "Demographics fit",
    "date": "Dating attitude",
    "go_out": "Dating attitude",
    "exphappy": "Dating attitude",
    "met_before": "Familiarity",
    "met_before_o": "Familiarity",
}
THEMES = sorted(set(THEME_MAP.values()))
assert set(S1_FEATS) <= set(THEME_MAP), "every Stage-1 feature needs a theme"

print(f"S1 feats: {len(S1_FEATS)} | kNN dims: {len(KNN_COLS)} | themes: {THEMES}")


S1 feats: 29 | kNN dims: 76 | themes: ['Ambition', 'Attractiveness', 'Dating attitude', 'Demographics fit', 'Familiarity', 'Fun', 'Intelligence', 'Shared interests', 'Sincerity']


## Cell 0-5 — Splitting schemes (S1–S10)


In [19]:
DECK_FOLDS = {
    "fold_1": [1, 4, 5, 17, 19, 20, 21],
    "fold_2": [3, 6, 7, 11, 13, 15, 18],
    "fold_3": [2, 8, 9, 10, 14, 16],
}
S2_CANON_CV = {
    "cv0": [2, 5, 6, 7, 11, 13],
    "cv1": [4, 8, 17, 20, 21],
    "cv2": [3, 10, 14, 16, 19],
}
HOLD_A, HOLD_B, HOLD_W19 = [1, 9, 15, 18], [8, 13, 14, 19], [19]


def make_3_folds(pool_waves, df):
    mr = (
        df[df["wave"].isin(pool_waves)]
        .groupby("wave")["match"]
        .mean()
        .sort_values()
    )
    folds = {0: [], 1: [], 2: []}
    for i, w in enumerate(mr.index.tolist()):
        folds[i % 3].append(int(w))
    return folds


def _cv_splits(folds: dict, tag_prefix="cv"):
    keys = list(folds)
    return [
        {
            "tag": f"{tag_prefix}{i}",
            "kind": "cv",
            "train_waves": sorted(w for j, k2 in enumerate(keys) if j != i for w in folds[k2]),
            "eval_waves": sorted(folds[k]),
        }
        for i, k in enumerate(keys)
    ]


def scheme_splits(scheme_id: str, df: pd.DataFrame):
    waves = sorted(int(w) for w in df["wave"].unique())

    def lowo(rotation_waves, all_waves):
        return [
            {
                "tag": f"wave{w}",
                "kind": "lowo",
                "train_waves": [x for x in all_waves if x != w],
                "eval_waves": [w],
            }
            for w in rotation_waves
        ]

    if scheme_id in ("S1", "S2"):
        return lowo(waves, waves)
    if scheme_id in ("S3", "S4"):
        folds = {k: list(v) for k, v in DECK_FOLDS.items()}
        if scheme_id == "S3":
            folds["fold_3"] = folds["fold_3"] + [12]
        return _cv_splits(folds, "fold")
    if scheme_id in ("S5", "S6"):
        folds = {k: list(v) for k, v in S2_CANON_CV.items()}
        if scheme_id == "S5":
            folds["cv2"] = folds["cv2"] + [12]
        pool = sorted(w for ws in folds.values() for w in ws)
        out = _cv_splits(folds)
        out.append({
            "tag": "holdout",
            "kind": "holdout",
            "train_waves": pool,
            "eval_waves": sorted(HOLD_A),
        })
        return out
    if scheme_id in ("S7", "S8"):
        pool = [w for w in waves if w not in HOLD_B]
        folds = make_3_folds(pool, df)
        out = _cv_splits({f"rr{k}": v for k, v in folds.items()})
        out.append({
            "tag": "holdout",
            "kind": "holdout",
            "train_waves": sorted(pool),
            "eval_waves": sorted(HOLD_B),
        })
        return out
    if scheme_id in ("S9", "S10"):
        rot = [w for w in waves if w not in HOLD_W19]
        out = lowo(rot, rot)
        out.append({
            "tag": "holdout_w19",
            "kind": "holdout",
            "train_waves": rot,
            "eval_waves": HOLD_W19,
        })
        return out
    raise ValueError(scheme_id)


SCHEME_VARIANT = {
    "S1": "keep", "S2": "drop", "S3": "keep", "S4": "drop",
    "S5": "keep", "S6": "drop", "S7": "keep", "S8": "drop",
    "S9": "keep", "S10": "drop",
}

for sid, var in SCHEME_VARIANT.items():
    df = DATA[var]
    waves = set(int(w) for w in df["wave"].unique())
    for sp in scheme_splits(sid, df):
        tr, ev = set(sp["train_waves"]), set(sp["eval_waves"])
        assert tr.isdisjoint(ev), f"{sid}/{sp['tag']}: train∩eval"
        assert tr | ev <= waves, f"{sid}/{sp['tag']}: unknown wave"
        sub = df[df["wave"].isin(ev)]
        pairs = set(map(tuple, sub[["iid", "pid"]].astype(int).values))
        sample = list(pairs)[:25]
        assert all((b, a) in pairs for a, b in sample), f"{sid}/{sp['tag']}: mirror pair missing"
    print(f"{sid} ({var}): {len(scheme_splits(sid, df))} splits OK")



S1 (keep): 21 splits OK
S2 (drop): 20 splits OK
S3 (keep): 3 splits OK
S4 (drop): 3 splits OK
S5 (keep): 4 splits OK
S6 (drop): 4 splits OK
S7 (keep): 4 splits OK
S8 (drop): 4 splits OK
S9 (keep): 21 splits OK
S10 (drop): 20 splits OK


## Cell 0-6 — Leakage-safe preprocessor


In [20]:
IMP_COLS = (
    CONT_COLS + SELF_COLS + NORM_COLS + BIN_COLS
    + SELF_B + INTEREST_B + NORM_B + ["int_cos"]
)
SCALE_COLS = CONT_COLS + INTEREST_B + ["int_cos"]


def fit_preprocessor(train_df: pd.DataFrame) -> dict:
    fills = train_df[IMP_COLS].median()
    scaler = StandardScaler().fit(train_df[SCALE_COLS].fillna(fills[SCALE_COLS]))
    t = train_df.copy()
    t[IMP_COLS] = t[IMP_COLS].fillna(fills)
    z = {}
    for g in (0, 1):
        a_rows, b_rows = t[t["gender"] == g], t[t["gender_B"] == g]
        for c in SELF_COLS:
            z[f"A|{g}|{c}"] = (float(a_rows[c].mean()), float(a_rows[c].std()))
            z[f"B|{g}|{c}"] = (
                float(b_rows[f"{c}_B"].mean()),
                float(b_rows[f"{c}_B"].std()),
            )
    return {"fills": fills, "scaler": scaler, "z": z}


def apply_preprocessor(frame: pd.DataFrame, pp: dict) -> pd.DataFrame:
    out = frame.copy()
    out[IMP_COLS] = out[IMP_COLS].fillna(pp["fills"])
    out[SCALE_COLS] = pp["scaler"].transform(out[SCALE_COLS])
    for side, gcol, suff in (("A", "gender", ""), ("B", "gender_B", "_B")):
        for c in SELF_COLS:
            zc = f"{c}{suff}_z"
            out[zc] = np.nan
            for g in (0, 1):
                mu, sd = pp["z"][f"{side}|{g}|{c}"]
                m = out[gcol] == g
                out.loc[m, zc] = ((out.loc[m, f"{c}{suff}"] - mu) / sd) if sd > 0 else 0.0
    for t_, pref, bz in zip(TRAITS, NORM_COLS[:5], SELF_B_Z):
        out[f"align_{t_}"] = out[pref] * out[bz]
    out["align_shar"] = out["shar1_1_norm"] * out["int_cos"]
    out["scale_type_code"] = (out["scale_type"] == "100pt").astype(int)
    return out


def process_split(df, train_waves, eval_waves):
    tr_raw = df[df["wave"].isin(train_waves)].copy()
    ev_raw = df[df["wave"].isin(eval_waves)].copy()
    pp = fit_preprocessor(tr_raw)
    return apply_preprocessor(tr_raw, pp), apply_preprocessor(ev_raw, pp), pp


_sp = scheme_splits("S8", DATA["drop"])[0]
_tr, _ev, _pp = process_split(DATA["drop"], _sp["train_waves"], _sp["eval_waves"])
_needed = set(S1_FEATS + KNN_COLS + CTX_COLS)
assert not _tr[list(_needed)].isna().any().any(), "NaN left in train features"
assert not _ev[list(_needed)].isna().any().any(), "NaN left in eval features"
print(
    f"smoke OK: train={len(_tr):,} eval={len(_ev):,} | "
    f"align_attr range [{_tr['align_attr'].min():.2f}, {_tr['align_attr'].max():.2f}]"
)




smoke OK: train=3,822 eval=2,604 | align_attr range [-2.08, 1.83]


## Cell 0-7 — Metrics


In [21]:
def reciprocal_scores(test_df, p):
    lut = {
        (int(i), int(j)): float(s)
        for (i, j), s in zip(zip(test_df["iid"], test_df["pid"]), p)
    }

    def hm(a, b):
        return 0.0 if (a + b) == 0 else 2 * a * b / (a + b)

    rev = [
        lut.get((int(j), int(i)), float(s))
        for (i, j, s) in zip(test_df["iid"], test_df["pid"], p)
    ]
    p = np.asarray(p, dtype=float)
    rev = np.asarray(rev, dtype=float)
    return {
        "uni": p,
        "recip_hm": np.array([hm(a, b) for a, b in zip(p, rev)]),
        "recip_gm": np.sqrt(p * rev),
        "rev": rev,
    }


def mirror_col(df, col):
    lut = {
        (int(i), int(j)): v
        for i, j, v in zip(df["iid"], df["pid"], df[col])
    }
    return np.array([
        lut.get((int(j), int(i)), v)
        for i, j, v in zip(df["iid"], df["pid"], df[col])
    ])


def ndcg_at5(df, score_col):
    vals = []
    for _, g in df.groupby("iid"):
        if g["match"].sum() == 0:
            continue
        vals.append(
            ndcg_score(g[["match"]].T.values, g[[score_col]].T.values, k=5)
        )
    return (float(np.mean(vals)) if vals else float("nan"), len(vals))


def prec_at_k(df, score_col, k):
    vals = []
    for _, g in df.groupby("iid"):
        if g["match"].sum() == 0:
            continue
        vals.append(g.nlargest(k, score_col)["match"].mean())
    return float(np.mean(vals)) if vals else float("nan")


def coverage_at5(df, score_col):
    top5 = {}
    for iid, g in df.groupby("iid"):
        top5[int(iid)] = set(g.nlargest(5, score_col)["pid"].astype(int))
    pairs = {
        tuple(sorted((int(a), int(b))))
        for a, b in df.loc[df["match"] == 1, ["iid", "pid"]].values
    }
    if not pairs:
        return float("nan")
    hit = sum(
        (b in top5.get(a, set())) or (a in top5.get(b, set()))
        for a, b in pairs
    )
    return hit / len(pairs)


print("Metrics ready (v1, reference-compatible).")
print("Foundation OK.")


Metrics ready (v1, reference-compatible).
Foundation OK.
